<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m1_02_numerical_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M1.02 — Predicting high wages from numbers

In the lab we trained our first models on the numerical columns of the
adult census, scaled the data, built a pipeline and used cross-validation.
In this workbook you do the same **on the 1985 wages survey**, and you will
see what happens when the **units** of a feature change.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "dominant_feature": "013f54400c82da08",
    "dummy_accuracy": "18db98e3760ea465",
    "logreg_accuracy": "5ecbc825ba5c1685",
    "n_train": "26d228663f13a885",
    "numerical_features": "8809d6cbbcfd05bc",
    "slower_version": "9322efcf93952a27"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data and target

Each row of the dataset is a person surveyed in 1985. We predict whether a
person earns **more than $10 an hour** (`WAGE` column). In this workbook we
only use the **numerical** columns: years of education, years of
experience and age.

✏️ Fill in `numerical_features` with the names of these three columns (see
the output of `wages.head()`), as a list of strings.

In [ ]:
import pandas as pd

wages = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/cps_85_wages.csv"
)
wages.head()

In [ ]:
numerical_features = ____
check("numerical_features", numerical_features)

data = wages[numerical_features]
target = wages["WAGE"] > 10
data.head()

`data` holds the features (one row per person, one column per feature) and
`target` holds the answer we want to predict (`True` / `False`).

## 2. Train and test sets

To know how well a model works on people it has **never seen**, we hide a
part of the data during training and use it only for testing.

✏️ Keep **a quarter** of the people for testing: set `test_size` to that
fraction (a number between 0 and 1). `random_state=42` makes the split the
same for everyone.

In [ ]:
from sklearn.model_selection import train_test_split

test_size = ____
data_train, data_test, target_train, target_test = train_test_split(
    data, target, test_size=test_size, random_state=42
)
print("train:", data_train.shape, "test:", data_test.shape)

✏️ How many people are in the **training** set?

In [ ]:
n_train = ____
check("n_train", n_train)

## 3. A first model: nearest neighbours

A k-nearest neighbours classifier predicts the class of a person by looking
at the `k` most similar people in the training set and taking the majority
answer. "Similar" means **close in distance**, computed from the feature
values.

✏️ Use **15** neighbours.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

n_neighbors = ____
knn = KNeighborsClassifier(n_neighbors=n_neighbors)
knn.fit(data_train, target_train)
knn.score(data_test, target_test)

`score` gives the **accuracy** on the test set: the share of people for
whom the prediction is right.

✏️ Write this accuracy rounded to 2 decimals.

In [ ]:
knn_accuracy = ____
check_score("knn_accuracy", knn_accuracy, knn.score(data_test, target_test))

### ➕ Going further: is this score good? Compare with a baseline

A **baseline** is a model that does not learn anything useful from the
features. Any real model must beat it. scikit-learn provides
`DummyClassifier`; with `strategy="most_frequent"` it always predicts the
most common class of the training set.

🔮 **Predict first.** Will the dummy model's accuracy be **below 0.5**,
**between 0.5 and 0.75**, or **above 0.75**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["below 0.5", "between 0.5 and 0.75", "above 0.75"])

✏️ Set `strategy` to the value described above (as a string).

In [ ]:
from sklearn.dummy import DummyClassifier

strategy = ____
dummy = DummyClassifier(strategy=strategy)
dummy.fit(data_train, target_train)
dummy.score(data_test, target_test)

In [ ]:
dummy_accuracy = ____
check("dummy_accuracy", dummy_accuracy)

🧠 **Think.** Why does a model that never looks at the data reach almost
70%? (Hint: what share of people earn more than $10?)

## 4. A linear model

Logistic regression is another classifier. It learns one **weight** per
feature and combines them into a probability.

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression()
logreg.fit(data_train, target_train)
logreg.score(data_test, target_test)

✏️ Accuracy of the logistic regression, rounded to 2 decimals:

In [ ]:
logreg_accuracy = ____
check("logreg_accuracy", logreg_accuracy)

## 5. ➕ Going further: what if the units change?

Imagine that the HR system we got the data from stores the **age in days**
instead of years. Nothing else changes: same people, same information.

In [ ]:
data_train_days = data_train.assign(AGE=data_train["AGE"] * 365)
data_test_days = data_test.assign(AGE=data_test["AGE"] * 365)
data_train_days.head()

🔮 **Predict first.** We train the same 15-neighbours model on this data.
Will the test accuracy go **"up"**, **"down"**, or stay **"the same"**
(compared to the score with age in years)?

In [ ]:
my_prediction = ____
commit(my_prediction, ["up", "down", "the same"])

In [ ]:
knn_days = KNeighborsClassifier(n_neighbors=15)
knn_days.fit(data_train_days, target_train)
knn_days.score(data_test_days, target_test)

✏️ Accuracy with age in days, rounded to 2 decimals. Compare it with the
**dummy** model.

In [ ]:
knn_days_accuracy = ____
check_score("knn_days_accuracy", knn_days_accuracy, knn_days.score(data_test_days, target_test))

**Why?** The model compares people with a distance: for each feature it
takes the difference between two people, squares it, and adds everything
up. Let us look at these squared differences for the first two people of
the training set:

In [ ]:
person_a, person_b = data_train_days.iloc[0], data_train_days.iloc[1]
pd.DataFrame(
    {"person A": person_a, "person B": person_b, "squared difference": (person_a - person_b) ** 2}
)

✏️ Which feature now decides almost alone who is "close" to whom?

In [ ]:
dominant_feature = ____
check("dominant_feature", dominant_feature)

A difference of 20 years of experience counts 400 in the distance, while a
difference of 20 years of age counts more than 53 million. The model
effectively ignores education and experience.

The fix is to put all features on a **comparable scale** before computing
distances. `StandardScaler` transforms each feature so that it has mean 0
and standard deviation 1. A **pipeline** chains the scaler and the model,
so the scaling learned on the training set is reused on the test set.

🔮 **Predict first.** With the scaler, will the accuracy (still age in days)
be **"below the dummy"**, **"close to the original score"**, or
**"far above the original score"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["below the dummy", "close to the original score", "far above the original score"])

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

scaled_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))
scaled_knn.fit(data_train_days, target_train)
scaled_knn.score(data_test_days, target_test)

In [ ]:
scaled_knn_days_accuracy = ____
check_score("scaled_knn_days_accuracy", scaled_knn_days_accuracy, scaled_knn.score(data_test_days, target_test))

### Does the logistic regression care about units?

🔮 **Predict first.** Logistic regression does not compute distances: it
multiplies each feature by a weight, and can simply learn a weight 365
times smaller for age in days. Will its accuracy with age in days be
**"much lower"**, **"about the same"** or **"much higher"** than 0.74?

In [ ]:
my_prediction = ____
commit(my_prediction, ["much lower", "about the same", "much higher"])

In [ ]:
logreg_days = LogisticRegression()
logreg_days.fit(data_train_days, target_train)
print("accuracy:", round(logreg_days.score(data_test_days, target_test), 3))

scaled_logreg_days = make_pipeline(StandardScaler(), LogisticRegression())
scaled_logreg_days.fit(data_train_days, target_train)
print("accuracy with scaling:", round(scaled_logreg_days.score(data_test_days, target_test), 3))

The accuracy hardly moves. But scaling still matters for this model: it
finds its weights step by step (**iterations**), and badly scaled features
make that search slower. `n_iter_` tells how many iterations were needed:

In [ ]:
print("iterations without scaling:", logreg_days.n_iter_[0])
print("iterations with scaling:   ", scaled_logreg_days[-1].n_iter_[0])

✏️ Which version needed **more** iterations? Write `"without scaling"` or
`"with scaling"`.

In [ ]:
slower_version = ____
check("slower_version", slower_version)

On a dataset with thousands of features and millions of rows, this
difference is the difference between seconds and hours, or a model that
stops before it has converged.

## 6. One split is not enough: cross-validation

Our test set holds only 134 people. With another random split, the scores
could be different. **Cross-validation** repeats the train/test procedure
on several splits and reports all the scores.

✏️ Use **5** splits (folds).

In [ ]:
from sklearn.model_selection import cross_validate

cv = ____
data_days = data.assign(AGE=data["AGE"] * 365)
models = {
    "dummy": DummyClassifier(strategy="most_frequent"),
    "scaled knn": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
    "scaled logreg": make_pipeline(StandardScaler(), LogisticRegression()),
}
results = {}
for name, model in models.items():
    scores = cross_validate(model, data_days, target, cv=cv)["test_score"]
    results[name] = scores
    print(f"{name:14s} {scores.round(2)}  mean {scores.mean():.2f} ± {scores.std():.2f}")

✏️ Mean cross-validated accuracy of the **scaled knn**, rounded to 2
decimals:

In [ ]:
cv_knn_accuracy = ____
check_score("cv_knn_accuracy", cv_knn_accuracy, results["scaled knn"].mean())

🧠 **Think.**

1. On our single split, the k-NN scored about 0.72–0.75. With
   cross-validation it scores less on average. Which number would you
   report to your boss, and why?
2. Look at the 5 scores of the k-NN: how much do they vary from fold to
   fold? Is the difference between the k-NN and the dummy model larger than
   this variation?
3. Which model would you pick, and how confident are you that it is really
   better than the dummy?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - why the k-NN broke when age was in days, and how scaling fixed it;
   - why the logistic regression did not care about units, and why scaling
     still helps it;
   - single split versus cross-validation: what changed and why it matters.